In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('cleaned_data.csv')
df

,Unnamed: 0,order_id,distance_km,item_count,peak_hour,weather,delivery_minutes,data_issue
0,0,O101,3.2,2.0,False,Clear,32.0,False
1,1,O102,5.5,4.0,True,Rain,48.0,False
2,2,O103,2.0,1.0,False,Clear,25.0,False
3,3,O104,6.1,3.0,True,Rain,42.0,False
4,4,O105,4.0,2.0,False,Cloudy,38.0,False
5,5,O106,8.0,5.0,True,Rain,61.0,False
6,6,O107,0.0,2.0,False,Clear,29.0,True
7,7,O108,3.8,0.0,True,cloudy,40.0,True
8,8,O109,7.2,4.0,True,Clear,55.0,False
9,9,O110,1.5,1.0,False,Clear,20.0,False


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,r2_score

In [4]:
features = [
    "distance_km",
    "item_count",
    "peak_hour",
    "weather"
]

target = "delivery_minutes"

# Exclude rows containing missing predictors or target values
model_df = df.dropna(
    subset=features + [target]
).copy()

print(model_df[["order_id"] + features + [target]])


   order_id  distance_km  item_count  peak_hour   weather  delivery_minutes
0      O101          3.2         2.0      False     Clear              32.0
1      O102          5.5         4.0       True      Rain              48.0
2      O103          2.0         1.0      False     Clear              25.0
3      O104          6.1         3.0       True      Rain              42.0
4      O105          4.0         2.0      False    Cloudy              38.0
5      O106          8.0         5.0       True      Rain              61.0
6      O107          0.0         2.0      False     Clear              29.0
7      O108          3.8         0.0       True   cloudy               40.0
8      O109          7.2         4.0       True     Clear              55.0
9      O110          1.5         1.0      False     Clear              20.0
10     O111          5.0         3.0       True   Unknown               0.0


In [5]:
X = model_df[features]
y = model_df[target]

In [6]:
X_train, X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

In [7]:
numeric_features = [
    "distance_km",
    "item_count"
]

boolean_features = [
    "peak_hour"
]

categorical_features = [
    "weather"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "boolean",
            "passthrough",
            boolean_features
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)



In [8]:
model = Pipeline(
    steps = [
        ("preprocessor",preprocessor),
        ("regressor",LinearRegression())
    ]
)

model.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('boolean', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "order_id": model_df.loc[X_test.index, "order_id"],
    "actual_minutes": y_test,
    "predicted_minutes": y_pred
})

results["absolute_error"] = (
    results["actual_minutes"]
    - results["predicted_minutes"]
).abs()

print(results)

  order_id  actual_minutes  predicted_minutes  absolute_error
5     O106            61.0          58.428571        2.571429
0     O101            32.0          33.571429        1.571429
9     O110            20.0          24.285714        4.285714


In [10]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

r2 = r2_score(
    y_test,
    y_pred
)

print(f"MAE: {mae:.2f} minutes")
print(f"R²: {r2:.2f}")

MAE: 2.81 minutes
R²: 0.97
